# Twenty-four-hour day-ahead scheduling

This notebook builds a 24-hour demand profile for the MATPOWER five-bus system, solves the complete classical reference, and compares it with an hourly QAOA decomposition. The quantum comparison uses one five-qubit commitment problem per hour and evaluates the resulting trajectory with the same dispatch and network checks.

## Model

For hour $t$, $u_{g,t}$ is the on/off state, $y_{g,t}$ is a start-up indicator, and $p_{g,t}$ is generation. The day-ahead objective is

$$\min \sum_{t=0}^{23}\left[\sum_g(a_g p_{g,t}^2+b_g p_{g,t})+\sum_g F_g u_{g,t}+\sum_g S_g y_{g,t}\right].$$

The hourly constraints include $\sum_g p_{g,t}=D_t$, $\sum_g P_g^{\max}u_{g,t}\ge D_t+R_t$, generator limits, the start-up relation $y_{g,t}\ge u_{g,t}-u_{g,t-1}$, and the five-bus DC branch-flow limits.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))

from case5_unit_commitment import load_case5_day_ahead, solve_milp_uc

instance = load_case5_day_ahead()
hours = np.arange(1, instance.time_periods + 1)
print(f'{instance.time_periods} hours; load range: {min(instance.demand_mw):.0f}--{max(instance.demand_mw):.0f} MW')


In [ ]:
reference = solve_milp_uc(instance, evaluate_method='linprog', enforce_network=True)
if not reference.success or not reference.schedule.success:
    raise RuntimeError(reference.message)
print(f'classical day-ahead cost: {reference.schedule.total_cost:.6f}')
print('peak hour:', int(np.argmax(instance.demand_mw) + 1))
print('maximum branch-flow violation:', max(d.max_line_violation_mw for d in reference.schedule.dispatch))


In [ ]:
plt.figure(figsize=(11, 3.5))
plt.plot(hours, instance.demand_mw, marker='o', linewidth=2, label='load')
plt.plot(hours, np.asarray(instance.demand_mw) + np.asarray(instance.reserve_mw), '--', label='load + reserve')
plt.xlabel('Hour of day')
plt.ylabel('MW')
plt.xticks(hours)
plt.grid(axis='y', alpha=0.25)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


## Hourly QAOA comparison

The following cell keeps each local circuit at five qubits. It is intended as a reproducible state-vector validation, not as a replacement for the joint 24-hour classical reference.

In [ ]:
from case5_unit_commitment import solve_hourly_qaoa

quantum = solve_hourly_qaoa(
    instance, backend='statevector', shots=128, maxiter=2, seed=11,
    evaluate_method='linprog', enforce_network=True,
)
gap = 100.0 * (quantum.schedule.total_cost - reference.schedule.total_cost) / reference.schedule.total_cost
print(f'hourly-QAOA day cost: {quantum.schedule.total_cost:.6f}')
print(f'relative gap: {gap:.4f}%')
print('fallback hours:', sum(item.fallback_to_classical for item in quantum.hourly_results))


The command-line driver additionally writes the complete JSON record and two publication-ready figures. From the repository root run:

    python scripts\\run_day_ahead.py --output results\\day_ahead

All computations in this notebook are local. No API key, cloud task, or real quantum processor is required.